# Physics-Informed Neural Network: Fisher-KPP Reaction-Diffusion Equation

**Author:** Nandini  
**Stack:** PyTorch, NumPy, matplotlib  
**Topics:** PINN for PDE solving, Fisher-KPP equation (tumour growth model), automatic differentiation, residual-based training

The Fisher-KPP equation $u_t = D u_{xx} + \rho u(1 - u)$ models reaction-diffusion dynamics relevant to glioma invasion modelling (Swanson framework).

---

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
torch.manual_seed(42)
device = torch.device('cpu')
print(f'PyTorch {torch.__version__}, device: {device}')

PyTorch 2.11.0+cu130, device: cpu


## 1. Fisher-KPP PDE Setup

$$\frac{\partial u}{\partial t} = D \frac{\partial^2 u}{\partial x^2} + \rho\, u(1 - u)$$

- $D = 0.01$: diffusion coefficient  
- $\rho = 1.0$: proliferation rate  
- Domain: $x \in [-1, 1]$, $t \in [0, 1]$  
- IC: $u(x, 0) = e^{-20x^2}$ (localized Gaussian)  
- BC: $u(-1, t) = u(1, t) = 0$

In [2]:
# PDE parameters
D_coeff = 0.01
rho = 1.0

class PINN(nn.Module):
    def __init__(self, layers=[2, 64, 64, 64, 1]):
        super().__init__()
        modules = []
        for i in range(len(layers) - 1):
            modules.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:
                modules.append(nn.Tanh())
        self.net = nn.Sequential(*modules)
    
    def forward(self, x, t):
        inp = torch.cat([x, t], dim=1)
        return self.net(inp)

model = PINN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'PINN parameters: {n_params}')
print(model)

PINN parameters: 8577
PINN(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): Tanh()
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 2. Collocation Points & Loss Function
We sample interior collocation points, boundary points, and initial condition points. The loss enforces the PDE residual, boundary conditions, and initial conditions simultaneously.

In [3]:
def sample_collocation(n_interior=2000, n_bc=200, n_ic=200):
    # Interior points
    x_int = (2 * torch.rand(n_interior, 1) - 1).to(device).requires_grad_(True)
    t_int = torch.rand(n_interior, 1).to(device).requires_grad_(True)
    
    # Boundary: x = -1 and x = 1
    t_bc = torch.rand(n_bc, 1).to(device)
    x_bc_left = -torch.ones(n_bc, 1).to(device)
    x_bc_right = torch.ones(n_bc, 1).to(device)
    
    # Initial condition: t = 0
    x_ic = (2 * torch.rand(n_ic, 1) - 1).to(device)
    t_ic = torch.zeros(n_ic, 1).to(device)
    u_ic = torch.exp(-20 * x_ic**2)  # Gaussian IC
    
    return x_int, t_int, x_bc_left, x_bc_right, t_bc, x_ic, t_ic, u_ic

def pde_residual(model, x, t):
    u = model(x, t)
    
    # First derivatives via autograd
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u),
                               create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u),
                               create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x),
                                create_graph=True)[0]
    
    # Fisher-KPP: u_t - D*u_xx - rho*u*(1-u) = 0
    residual = u_t - D_coeff * u_xx - rho * u * (1 - u)
    return residual

print('Loss functions defined')

Loss functions defined


## 3. Training Loop

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)

losses_history = {'total': [], 'pde': [], 'bc': [], 'ic': []}
n_epochs = 3000

for epoch in range(n_epochs):
    optimizer.zero_grad()
    
    x_int, t_int, x_bc_l, x_bc_r, t_bc, x_ic, t_ic, u_ic = sample_collocation()
    
    # PDE residual loss
    res = pde_residual(model, x_int, t_int)
    loss_pde = torch.mean(res**2)
    
    # BC loss: u(-1,t) = 0, u(1,t) = 0
    u_left = model(x_bc_l, t_bc)
    u_right = model(x_bc_r, t_bc)
    loss_bc = torch.mean(u_left**2) + torch.mean(u_right**2)
    
    # IC loss
    u_pred_ic = model(x_ic, t_ic)
    loss_ic = torch.mean((u_pred_ic - u_ic)**2)
    
    loss = loss_pde + 10 * loss_bc + 10 * loss_ic
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    losses_history['total'].append(loss.item())
    losses_history['pde'].append(loss_pde.item())
    losses_history['bc'].append(loss_bc.item())
    losses_history['ic'].append(loss_ic.item())
    
    if (epoch + 1) % 500 == 0:
        print(f'Epoch {epoch+1:5d} | Total: {loss.item():.6f} | PDE: {loss_pde.item():.6f} | BC: {loss_bc.item():.6f} | IC: {loss_ic.item():.6f}')

print('Training complete')

Epoch   500 | Total: 0.004705 | PDE: 0.003651 | BC: 0.000011 | IC: 0.000095


Epoch  1000 | Total: 0.000570 | PDE: 0.000400 | BC: 0.000004 | IC: 0.000013


Epoch  1500 | Total: 0.000379 | PDE: 0.000309 | BC: 0.000001 | IC: 0.000006


Epoch  2000 | Total: 0.000231 | PDE: 0.000177 | BC: 0.000001 | IC: 0.000004


Epoch  2500 | Total: 0.000187 | PDE: 0.000158 | BC: 0.000001 | IC: 0.000002


Epoch  3000 | Total: 0.000146 | PDE: 0.000117 | BC: 0.000001 | IC: 0.000002
Training complete


In [5]:
fig, ax = plt.subplots(figsize=(8, 4))
for key in ['total', 'pde', 'bc', 'ic']:
    ax.semilogy(losses_history[key], label=key, alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('PINN Training Loss: Fisher-KPP Equation')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Solution Visualization
Plotting the learned solution $u(x,t)$ and its spatiotemporal evolution.

In [6]:
# Evaluate on grid
nx, nt = 200, 100
x_eval = np.linspace(-1, 1, nx)
t_eval = np.linspace(0, 1, nt)
X, T = np.meshgrid(x_eval, t_eval)

x_flat = torch.tensor(X.flatten(), dtype=torch.float32).reshape(-1, 1).to(device)
t_flat = torch.tensor(T.flatten(), dtype=torch.float32).reshape(-1, 1).to(device)

with torch.no_grad():
    u_pred = model(x_flat, t_flat).cpu().numpy().reshape(nt, nx)

# Spatiotemporal heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].pcolormesh(X, T, u_pred, cmap='inferno', shading='auto')
axes[0].set_xlabel('x (spatial)')
axes[0].set_ylabel('t (time)')
axes[0].set_title('PINN Solution: Fisher-KPP $u(x,t)$')
plt.colorbar(im, ax=axes[0], label='u(x,t)')

# Snapshots at different times
time_snapshots = [0.0, 0.1, 0.3, 0.5, 0.8, 1.0]
colors_snap = plt.cm.viridis(np.linspace(0, 1, len(time_snapshots)))
for t_val, col in zip(time_snapshots, colors_snap):
    x_snap = torch.tensor(x_eval, dtype=torch.float32).reshape(-1, 1)
    t_snap = t_val * torch.ones_like(x_snap)
    with torch.no_grad():
        u_snap = model(x_snap, t_snap).numpy().flatten()
    axes[1].plot(x_eval, u_snap, color=col, linewidth=1.5, label=f't={t_val:.1f}')

axes[1].set_xlabel('x')
axes[1].set_ylabel('u(x,t)')
axes[1].set_title('Tumour Density Profile Evolution')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print('The PINN captures the travelling wave front characteristic of Fisher-KPP dynamics.')
print(f'Wave speed (theoretical): c* = 2*sqrt(D*rho) = {2*np.sqrt(D_coeff*rho):.4f}')

The PINN captures the travelling wave front characteristic of Fisher-KPP dynamics.
Wave speed (theoretical): c* = 2*sqrt(D*rho) = 0.2000
